# Scomporre la Variabilità nella Liquidazione dei Sinistri lungo la Gerarchia Organizzativa di un Assicuratore con PROC NESTED

## Sintesi

Un assicuratore ramo danni vuole sapere **dove** ha origine l'incoerenza
nella liquidazione dei sinistri: è determinata da differenze tra
regioni geografiche, tra filiali, tra singoli liquidatori, o
semplicemente dal rumore da sinistro a sinistro? Questo notebook
costruisce un portafoglio sintetico e completamente annidato di dati
di sinistri auto (Regione > Filiale > Liquidatore > sinistro) e
utilizza **PROC NESTED** per eseguire un'analisi della varianza a
effetti casuali, stimando la componente di varianza attribuita a
ciascun livello della gerarchia e riportandola come percentuale del
totale. Il risultato indica alla direzione quale livello organizzativo
colpire per migliorare la coerenza delle liquidazioni.

## Fonti dei dati

Tutti i dati sono generati in linea dal primo DATA step — nessun file
esterno, nessuna rete. Il disegno è **completamente annidato**: ogni
filiale appartiene esattamente a una regione, ogni liquidatore
esattamente a una filiale, e ogni sinistro esattamente a un
liquidatore.

| Dataset | Righe | Variabile | Tipo | Ruolo | Descrizione |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | char | CLASS (livello 1, più esterno) | Regione geografica (5 regioni: NE, SE, CE, NO, SO) |
| | | `Branch` | char | CLASS (livello 2, annidato in Region) | Codice di filiale (2 filiali per regione) |
| | | `Adjuster` | char | CLASS (livello 3, annidato in Branch) | ID del liquidatore sinistri (2 liquidatori per filiale) |
| | | `Settlement` | num | VAR (dipendente) | Indennizzo pagato sul sinistro auto, in USD |
| | | `CycleDays` | num | VAR (dipendente) | Giorni dalla prima notifica del sinistro alla liquidazione |

Struttura sintetica: 5 regioni x 2 filiali x 2 liquidatori x 2 sinistri
= 40 osservazioni. Un effetto casuale di regione, un effetto casuale
di filiale annidato nella regione, un effetto casuale di liquidatore
annidato nella filiale, e un residuo a livello di singolo sinistro
sono sovrapposti in modo additivo tramite `rand('NORMAL', ...)`
cosicché le componenti di varianza siano recuperabili. L'effetto
regione è generato con la deviazione standard più grande (2.200)
affinché *dove* viene gestito un sinistro sia il fattore dominante.
Seed fissato con `call streaminit(20260531)`. Un disegno compatto e
completamente bilanciato mantiene ogni livello della gerarchia
popolato con gradi di libertà reali.

# Scomporre la Variabilità nella Liquidazione dei Sinistri con PROC NESTED

Quando un assicuratore ramo danni analizza quanto paga per liquidare i
sinistri auto, spesso trova un'ampia variazione. Parte di questa
variazione è inevitabile (ogni incidente è diverso), ma una parte
riflette fattori **organizzativi** — una regione liquida in modo più
generoso di un'altra, una filiale è più generosa, un singolo
liquidatore è un valore anomalo.

I dati hanno una struttura strettamente **annidata** (gerarchica):

```
Regione  ->  Filiale (annidata in Regione)  ->  Liquidatore (annidato in Filiale)  ->  singoli sinistri
```

Una filiale appartiene esattamente a una regione e un liquidatore
esattamente a una filiale, quindi i fattori sono annidati anziché
incrociati. `PROC NESTED` esegue un'analisi della varianza a effetti
casuali proprio per questo disegno e stima una **componente di
varianza** a ciascun livello, espressa come percentuale del totale.
Quella ripartizione percentuale risponde alla domanda di business:
*quale livello dell'organizzazione dovremmo colpire per rendere le
liquidazioni più coerenti?*

## Passo 1 — Generare un portafoglio sintetico e completamente annidato di sinistri

Simuliamo 5 regioni, ciascuna con 2 filiali, ciascuna con 2
liquidatori, ciascuno che gestisce 2 sinistri (40 righe totali).
Costruiamo la risposta a partire da effetti casuali additivi cosicché
ogni livello contribuisca realmente alla varianza:

- un effetto di **regione** (dispersione maggiore — le regioni
  differiscono di più),
- un effetto di **filiale annidata nella regione**,
- un effetto di **liquidatore annidato nella filiale**,
- e un **residuo a livello di sinistro** (il rumore entro il singolo
  liquidatore).

`call streaminit` fissa il seed per la riproducibilità; ogni effetto è
generato con `rand('NORMAL', media, ds)`. L'effetto regione usa la
deviazione standard più ampia, quindi dovrebbe rivendicare la quota
maggiore della varianza totale. Generiamo anche una seconda risposta,
`CycleDays`, che condivide la stessa gerarchia cosicché si possa
dimostrare in seguito l'analisi multi-risposta.

In [1]:
DATI claims;
   CHIAMARE streaminit(20260531);
   LUNGHEZZA Region $2 Branch $6 Adjuster $9;

   /* Effetto casuale a livello di regione: le regioni differiscono di piu'. */
   FARE r = 1 FINO_A 5;
      Region = SCAN('NE SE CE NO SO', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* Filiale annidata nella regione. */
      FARE b = 1 FINO_A 2;
         Branch = catx('-', Region, PUT(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* Liquidatore annidato nella filiale. */
         FARE a = 1 FINO_A 2;
            Adjuster = catx('-', Branch, PUT(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* Singoli sinistri gestiti da questo liquidatore. */
            FARE claim = 1 FINO_A 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               SE_COND Settlement < 0 ALLORA Settlement = 0;
               SE_COND CycleDays  < 1 ALLORA CycleDays  = 1;
               USCITA;
            FINE;
         FINE;
      FINE;
   FINE;

   MANTENERE Region Branch Adjuster Settlement CycleDays;
ESEGUIRE;



NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Passo 2 — Ordinare secondo la gerarchia di annidamento

`PROC NESTED` richiede che i dati in input siano **ordinati secondo le
variabili CLASS nell'ordine in cui verranno elencate** — prima il
fattore più esterno. Ordiniamo per `Region Branch Adjuster` cosicché
la procedura possa percorrere correttamente la gerarchia.

In [2]:
PROCEDURA ORDINARE DATI=claims;
   PER Region Branch Adjuster;
ESEGUIRE;



NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## Passo 3 — Analisi delle componenti di varianza dell'importo di liquidazione

L'analisi centrale. Elenchiamo le variabili CLASS dalla più esterna
alla più interna (`Region Branch Adjuster`); la replicazione più
interna — i singoli sinistri — è trattata automaticamente come termine
di errore. `VAR Settlement` indica l'unica risposta.

L'istruzione `LABEL` fornisce un nome di visualizzazione più leggibile
per la risposta nel banner di output. L'output produce i coefficienti
dei quadrati medi attesi, la tabella di analisi della varianza (con un
test F di ciascuna componente di varianza contro zero), e la stima
della **componente di varianza** a ciascun livello insieme alla sua
**percentuale del totale**.

In [3]:
TITOLO 'Componenti di Varianza delle Liquidazioni dei Sinistri Auto';
title2 'ANOVA a Effetti Casuali Regione / Filiale / Liquidatore';

PROCEDURA nested DATI=claims;
   CLASSE Region Branch Adjuster;
   VARIABILE Settlement;
   ETICHETTA Region='Regione' Branch='Filiale' Adjuster='Liquidatore'
         Settlement = 'Indennizzo Pagato (USD)';
ESEGUIRE;


                              Componenti di Varianza delle Liquidazioni dei Sinistri Auto                               
                                ANOVA a Effetti Casuali Regione / Filiale / Liquidatore                                 


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Indennizzo Pagato (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Regione                4  62114122.126866          6.71      0.0304   Filiale  15528530.531717  1651824.844989             54.4057      8.0000
Filiale                5  11569658.859028          1.89      0.1829  Liquidatore  2313931.771806   272682.828942              8.9813      4.0000
Liquidatore           10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697


NOTE: Option TITLE changed to Componenti di Varianza delle Liquidazioni dei Sinistri Auto.
NOTE: Option TITLE2 changed to ANOVA a Effetti Casuali Regione / Filiale / Liquidatore.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Passo 4 — Analizzare due risposte contemporaneamente

Anche il **tempo di ciclo** interessa alla direzione — quanto tempo
impiegano i sinistri a essere liquidati. Aggiungiamo `CycleDays`
all'elenco `VAR`. Con più di una variabile dipendente, `PROC NESTED`
riporta inoltre un'**analisi di covariazione** che mostra come le due
risposte covariano a ciascun livello della gerarchia (per esempio, se
le regioni che pagano di più liquidano anche più lentamente).

In [4]:
TITOLO 'Componenti di Varianza di Liquidazione e Tempo di Ciclo';
title2 'Due Risposte sulla Stessa Gerarchia Annidata';

PROCEDURA nested DATI=claims;
   CLASSE Region Branch Adjuster;
   VARIABILE Settlement CycleDays;
   ETICHETTA Region='Regione' Branch='Filiale' Adjuster='Liquidatore'
         Settlement = 'Indennizzo Pagato (USD)'
         CycleDays  = 'Giorni per la Liquidazione';
ESEGUIRE;


                                Componenti di Varianza di Liquidazione e Tempo di Ciclo                                 
                                      Due Risposte sulla Stessa Gerarchia Annidata                                      


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Indennizzo Pagato (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Regione                4  62114122.126866          6.71      0.0304   Filiale  15528530.531717  1651824.844989             54.4057      8.0000
Filiale                5  11569658.859028          1.89      0.1829  Liquidatore  2313931.771806   272682.828942              8.9813      4.0000
Liquidatore           10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697


NOTE: Option TITLE changed to Componenti di Varianza di Liquidazione e Tempo di Ciclo.
NOTE: Option TITLE2 changed to Due Risposte sulla Stessa Gerarchia Annidata.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Passo 5 — Vista solo-varianza con l'opzione AOV

Per un riepilogo rapido delle componenti di varianza su entrambe le
risposte, l'opzione `AOV` limita l'output alle statistiche di analisi
della varianza e **sopprime la sezione di analisi di covariazione**.
Questa è la vista compatta che un analista di portafoglio scorrerebbe
quando ha bisogno solo della ripartizione della varianza per livello
per ciascuna risposta, non della covariazione incrociata tra
risposte.

In [5]:
TITOLO 'Solo Componenti di Varianza (AOV)';

PROCEDURA nested DATI=claims aov;
   CLASSE Region Branch Adjuster;
   VARIABILE Settlement CycleDays;
   ETICHETTA Region='Regione' Branch='Filiale' Adjuster='Liquidatore'
         Settlement = 'Indennizzo Pagato (USD)'
         CycleDays  = 'Giorni per la Liquidazione';
ESEGUIRE;

TITOLO;


                                           Solo Componenti di Varianza (AOV)                                            
                                      Due Risposte sulla Stessa Gerarchia Annidata                                      


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Indennizzo Pagato (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Regione                4  62114122.126866          6.71      0.0304   Filiale  15528530.531717  1651824.844989             54.4057      8.0000
Filiale                5  11569658.859028          1.89      0.1829  Liquidatore  2313931.771806   272682.828942              8.9813      4.0000
Liquidatore           10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697


NOTE: Option TITLE changed to Solo Componenti di Varianza (AOV).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Interpretazione dei risultati

La colonna **Percentuale del Totale** nella tabella di analisi della varianza è il dato principale. Leggendola dall'alto verso il basso si ottiene la quota di variabilità totale delle liquidazioni attribuibile a ciascun livello organizzativo. Per l'importo di liquidazione l'esecuzione produce:

- **Regione — 54.4%** (Pr > F = 0.0304). I dati sono stati generati con la dispersione più ampia a livello di regione, e l'analisi la recupera correttamente: più della metà di tutta la variabilità delle liquidazioni è *tra* le regioni, un'evidenza statisticamente significativa che *dove* viene gestito un sinistro è il fattore dominante.
- **Filiale (entro Regione) — 9.0%** (Pr > F = 0.18). Una quota aggiuntiva modesta quando si scende dalla regione alla singola filiale; non significativa in questo caso.
- **Liquidatore (entro Filiale) — 3.7%** (Pr > F = 0.33). Poca variazione è attribuibile al singolo liquidatore in questo portafoglio.
- **Errore — 32.9%.** Il rumore residuo da sinistro a sinistro entro un liquidatore; questa è la variazione irriducibile che nessuna leva organizzativa può eliminare.

Ogni livello porta anche un **test F (Pr > F)** dell'ipotesi nulla che la sua componente di varianza sia zero. Il piccolo p-value della Regione (0.0304) è un'evidenza statistica di differenze sistematiche genuine tra le regioni, non dovute al caso campionario.

La risposta sul tempo di ciclo racconta una storia parallela: **la Regione spiega il 45.8%** della variazione nei giorni-alla-liquidazione (Pr > F = 0.0448, ancora significativo), con Filiale e Liquidatore che contribuiscono quote a una cifra e il residuo che porta il 40.1%. Quindi sia *quanto* paga un assicuratore sia *quanto tempo* impiega sono governati anzitutto dalla regione.

L'esecuzione a due risposte aggiunge un'**analisi di covariazione**. Qui il livello Regione guida i prodotti incrociati, e il **coefficiente di correlazione complessivo è -0.36** — una relazione negativa: le regioni che pagano liquidazioni maggiori tendono a *chiuderle più rapidamente*, non più lentamente. Questo è un risultato utile e non ovvio — le regioni più costose non sono quelle più lente.

**Indicazione di business:** poiché la Regione domina la ripartizione percentuale del totale per entrambe le risposte, l'assicuratore dovrebbe standardizzare le linee guida di liquidazione e i limiti di autorità *tra le regioni* per primi — è lì che risiede l'incoerenza più ampia e statisticamente significativa — prima di investire in interventi a livello di filiale o di liquidatore. La correlazione negativa tra importo di liquidazione e tempo di ciclo significa che non esiste un'unica regione che sia sia la più costosa sia la più lenta, quindi i due problemi possono essere affrontati con programmi separati mirati per regione. PROC NESTED trasforma una vaga sensazione di "le nostre liquidazioni sono incoerenti" in un'attribuzione azionabile, livello per livello, di dove risiede tale incoerenza.